# 03 · MemorySaver — in-session memory

Without memory each `invoke` is independent — the agent has no idea what was said before.

With `MemorySaver` the state persists across invocations via `thread_id`.

> **Note:** `MemorySaver` lives in RAM — it is lost on restart. For real persistence use `SqliteSaver` or `PostgresSaver`.

In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

## Agent with memory

In [ ]:
@tool
def check_logs(component: str) -> str:
    """Checks error logs for a given pipeline component."""
    logs = {
        "retriever": "ERROR: retriever timeout after 30s — embedding service unreachable",
        "loader": "WARN: 3 documents skipped due to encoding errors",
        "llm": "OK: no errors in last 24h",
    }
    return logs.get(component.lower(), f"No logs found for '{component}'")


llm = ChatOpenAI(model="gpt-4o-mini")
memory = MemorySaver()
graph = create_react_agent(llm, tools=[check_logs], checkpointer=memory)

## Multi-turn conversation (same thread_id)

In [ ]:
config = {"configurable": {"thread_id": "support-001"}}

# Turn 1
r1 = graph.invoke(
    {"messages": [{"role": "user", "content": "I have an error in my RAG pipeline"}]},
    config,
)
print("Turn 1:", r1["messages"][-1].content)
print("---")

In [ ]:
# Turn 2
r2 = graph.invoke(
    {"messages": [{"role": "user", "content": "The error is in the retriever"}]},
    config,
)
print("Turn 2:", r2["messages"][-1].content)
print("---")

In [ ]:
# Turn 3 — should remember RAG + retriever
r3 = graph.invoke(
    {"messages": [{"role": "user", "content": "How do I fix it?"}]},
    config,
)
print("Turn 3:", r3["messages"][-1].content)

## Different thread → fresh conversation with no memory

In [ ]:
# Different thread — no prior context
config_new = {"configurable": {"thread_id": "support-002"}}

r_new = graph.invoke(
    {"messages": [{"role": "user", "content": "How do I fix it?"}]},
    config_new,
)
print("New thread:", r_new["messages"][-1].content)